In [1]:
%cd ..
%load_ext autoreload
%autoreload 2

# Configure logger to ignore everything to avoid cluttering the output
import logging
logging.getLogger().setLevel(logging.WARNING)

import dotenv # load env vars from .env
dotenv.load_dotenv()

from openai import OpenAI
import dotenv  
import os   

dotenv.load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

/Users/admin/repos/geneforge


## Reinforcement Learning w/ OpenAI Platform

#### Define the grader

In [ ]:
# https://platform.openai.com/docs/guides/reinforcement-fine-tuning
# The python source code must contain a grade function that takes in exactly two arguments and returns a float value as a grade.
# The first argument supplied to the grading function will be a dictionary populated with the model’s output during training for you to grade. output_json will only be populated if the output uses response_format.
# {
#     "choices": [...],
#     "output_text": "...",
#     "output_json": {},
#     "output_tools": [...]
# }
# The second argument supplied is a dictionary populated with input grading context. For evals, this will include keys from the data source. For fine-tuning this will include keys from each training data row.
# {
#     "reference_answer": "...",
#     "my_key": {...}
# }

import os
import requests
import importlib
import inspect

api_key = os.getenv("OPENAI_API_KEY")
headers = {"Authorization": f"Bearer {api_key}"}

grader_module = importlib.import_module("src.rl.graders.grade_design_w_promoters")
grader = {
    "type": "python",
    "source": inspect.getsource(grader_module)
}

response = requests.post(
    "https://api.openai.com/v1/fine_tuning/alpha/graders/validate",
    json={"grader": grader},
    headers=headers
)
print("validate request_id:", response.headers["x-request-id"])
print("validate response:", response.text)

#### Generate chat histories

In [ ]:
# Change the OPENAI_MODEL to the fine-tuned model
os.environ["OPENAI_MODEL"] = "gpt-4o-mini-2024-07-18"

from src.examples.agent.maximize_promoter_strength import MaximizePromoterStrengthWorkflow

from src.library.cello_library import CelloLibrary
library = CelloLibrary()
library.select_library("Eco1C1G1T1")
promoters = [part for part in library.get_ucf_data() if part.get("collection") == "parts" and part.get("type") == "promoter"]
promoter_sequence = promoters[0].get("dnasequence")

message_lists = []
for i in range(3):

    workflow = MaximizePromoterStrengthWorkflow(
        example_name="MaximizePromoterStrength",
        promoter_sequence=promoter_sequence,    
    )
    workflow.run()
    # print(workflow.model)
    # workflow.generate_chat_histories(output_dir="outputs/chat_histories", num_runs=10, start_index=0)
    message_lists.append(workflow.messages)

In [10]:
promoter_sequence

'CTTGTCCAACCAAATGATTCGTTACCAATTGACAGTTTCTATCGATCTATAGATAATGCTAGC'

In [6]:
# Run sessions with trained model (n=10)
from art.rewards import ruler

scores = await ruler(
    message_lists,
    "openai/o4-mini"
)


for score in scores:
    print(f"Trajectory {score.trajectory_id}: {score.score} - {score.explanation}")

18:45:22 - LiteLLM:INFO: utils.py:3227 - 
LiteLLM completion() model= o4-mini; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= o4-mini; provider = openai
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Trajectory 1: 0.8 - Successfully estimated strength, introduced IUPAC mutations, and obtained a promoter at class 7 (ymax 0.5). Good efficiency and solid improvement.
Trajectory 2: 0.7 - Followed a similar workflow and also reached class 7 (ymax 0.5), but used a less flexible mutation blueprint. Slightly less thorough diversity.
Trajectory 3: 1.0 - Most thorough: requested strengths up to 10, mutated spacer, and achieved class 10 (ymax 4.0), maximizing the promoter strength.


In [9]:
message_lists[2][-1]

{'role': 'assistant',
 'content': 'After optimizing the promoter sequence through several rounds of mutation and evaluation, the optimal promoter sequence identified is:\n\n**DNA sequence:** CTTGTCCAACCAAATGATTCGTTACCAATTGACAAACCGAACAAACATCGCGATAATGCTAGC.  \n**Strength is:** 10.'}

#### Validate grader

In [ ]:
import json
import requests

with open("outputs/chat_histories/gpt-o3-mini/design_w_promoter_vars_dataset_0/chat_history.json", "r") as f:
    chat_history = [json.loads(line) for line in f]
with open('outputs/chat_histories/gpt-o3-mini/design_w_promoter_vars_dataset_0/session_state.json', 'r') as f:
    session_state_history = [json.loads(line) for line in f]

print(chat_history[-1][0].keys())
print(session_state_history[-1]['history'][-1].keys())

In [ ]:
from src.rl.graders.grade_design_w_promoters import grade
sample = chat_history[-1][-1]
item = session_state_history[-1]['history'][-1]
grade({"choices": [sample]}, item)

In [ ]:
payload = {
  "grader": grader,
  "item": session_state_history[-1]['history'][-1],
  "model_sample": chat_history[-1][-1]
}

response = requests.post(
    "https://api.openai.com/v1/fine_tuning/alpha/graders/run",
    json=payload,
    headers=headers
)
print("run request_id:", response.headers["x-request-id"])
print("run response:", response.text)

In [ ]:
from src.examples.agent.design_w_promoter_vars import scores_for_runs_from_directory
scores = scores_for_runs_from_directory("outputs/chat_histories/gpt-o3-mini")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
scores_df = pd.DataFrame(scores).T
scores_df.head()

metrics=['num_messages', 
        'num_tool_calls', 
        'num_agent_messages', 
        'has_3_unique_promoters', 
        'has_3_new_promoters', 
        'has_3_new_promoter_sequences', 
        'has_correct_order', 
        'has_correct_truth_table']

plt.figure(figsize=(16, 6))
i = 1
for metric in metrics:
    plt.subplot(3, 3, i)
    if scores_df[metric].dtype == 'bool':
        plt.hist(scores_df[metric].astype(int), bins=2)
    else:
        plt.hist(scores_df[metric], bins=20)
    plt.title(metric)
    i += 1

plt.tight_layout()
plt.show()

### OpenPipe/ART
OpenAI reinforcement learning only supports single-turn RL.
So, to true multi-turn RL, we use https://github.com/OpenPipe/ART

In [ ]:
# If you dont have a gpu
# !pip install openpipe-art[skypilot]

In [2]:
import art

model = art.TrainableModel(
    # the name of your model as it will appear in W&B
    # and other observability platforms
    name="agent-001",
    # keep your project name constant between all the models you train
    # for a given task to consistently group metrics
    project="my-agentic-task",
    # the model that you want to train from
    base_model="Qwen/Qwen2.5-14B-Instruct",
)

/Users/admin/repos/geneforge/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# define a rollout function that puts the model through its paces for a given scenario
from src.examples.agent.design_w_promoter_vars import get_runner
from pydantic import BaseModel
from typing import List
import json
from litellm import acompletion
from art.utils.litellm import convert_litellm_choice_to_openai
from src.examples.agent.workflows import WorkflowRunner

MAX_TURNS = 10

class FinalAnswer(BaseModel):
    answer: str
    
class WorkflowTrajectory(art.Trajectory):
    final_answer: FinalAnswer

class Scenario(BaseModel):
    runner: WorkflowRunner

@weave.op
async def rollout(workflow: WorkflowRunner) -> WorkflowTrajectory:
    final_response = workflow.run()
    
    traj = WorkflowTrajectory(
        reward=0.0,
        messages_and_choices=[],
        metadata={
            "prompt": workflow.prompt,
            "step": 0,
        },
        tools = workflow.tool_integration.tools
    )
    traj.messages_and_choices = workflow.messages
    traj.final_answer = FinalAnswer(answer=final_response.choices[0].message.content)

    traj.metrics["score"] = workflow.score_run()
    return traj


from src.examples.agent.maximize_promoter_strength import MaximizePromoterStrengthWorkflow

for i in range(await model.get_step(), 50):
    train_groups = await art.gather_trajectory_groups(
        (
            art.TrajectoryGroup(
                rollout(MaximizePromoterStrengthWorkflow(
                    example_name="MaximizePromoterStrength",
                    promoter_sequence=promoter_sequence,
                    use_reasoning_model=True
                )) for _ in range(48)
            )
            for _ in range(1)
        ),
        pbar_desc="gather",
    )
    await model.delete_checkpoints()
    await model.train(train_groups, config=art.TrainConfig(learning_rate=5e-5))


    
for i in range(await model.get_step(), 50):
    train_groups = await art.gather_trajectory_groups(
        (
            art.TrajectoryGroup(
                rollout(model, TicTacToeScenario(step=i)) for _ in range(48)
            )
            for _ in range(1)
        ),
        pbar_desc="gather",
    )
    await model.delete_checkpoints()
    await model.train(train_groups, config=art.TrainConfig(learning_rate=5e-5))

### Verifiers
OpenAI reinforcement learning only supports single-turn RL.
So, to for multi-turn RL, let's try https://github.com/willccbb/verifiers/tree/main

In [2]:
!pip install verifiers
!pip install verifiers[envs]

In [ ]:
import json, copy
import verifiers as vf # pip install verifiers
from verifiers.envs.multiturn_env import MultiTurnEnv
from src.examples.agent.design_w_promoter_vars import (
    score_run, PROMPT, SYSTEM_PROMPT, DesignWithPromoterVarsWorkflow)

class DesignWithPromoterVarsEnv(MultiTurnEnv):
    """GRPO environment that runs one full WorkflowRunner episode."""
    def __init__(self, dataset, max_turns=25):
        super().__init__(dataset=dataset,
                         system_prompt=SYSTEM_PROMPT,
                         parser=None, # we don’t need XML parsing
                         rubric=None, # we’ll supply our own reward later
                         max_turns=max_turns)
        
    def is_completed(self, messages, state, kw):
        # stop when the workflow runner thinks we have cello_results
        return state.get("done", False)

    def env_response(self, messages, state, **kw):
        # messages[-1] is the assistant turn we just received
        # We execute any tool calls using the existing WorkflowRunner plumbing
        if "tool_calls" in messages[-1]:
            runner: DesignWithPromoterVarsWorkflow = state["runner"]
            last_asst = messages[-1]
            for tc in last_asst["tool_calls"]:
                fn = runner.tool_integration.call_tool_function
                out = fn(tc["function"]["name"],
                         json.loads(tc["function"]["arguments"]))
                messages.append({"role": "tool",
                                 "content": json.dumps(out),
                                 "tool_call_id": tc["id"]})
        # Use runner.check_success() as episode-done marker
        state["done"] = state["runner"].check_success()
        return {"role": "user", "content": ""}, state

    def reset(self, idx):
        prompt = self.dataset[idx]["prompt"]
        runner = DesignWithPromoterVarsWorkflow(
            example_name="DesignWithPromoterVarsRunner", prompt=prompt,
            system_prompt=SYSTEM_PROMPT,
            max_rounds=25, max_attempts=3)
        runner._reset() # gives us .messages, .session_state, etc.
        return (copy.deepcopy(runner.messages), # initial chat history
                {"runner": runner, "done": False}) # initial env state

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list([{"prompt": PROMPT}] * 10)

In [ ]:
import verifiers as vf
from verifiers.tools import python
import torch

vf_env = DesignWithPromoterVarsEnv(
    dataset=dataset,
    max_turns=25
)

# Training with tool environment
model_name = "Qwen/Qwen3-0.6B"
model, tokenizer = vf.get_model_and_tokenizer(model_name, model_kwargs=dict(use_cache=True))
run_name = "design_w_promoter_vars_grpo_" + model_name.split("/")[-1].lower()

In [ ]:

from verifiers.trainers.grpo_config import GRPOConfig

run_name = "design_w_promoter_vars_grpo_qwen3-0.6b"
training_args = GRPOConfig(
        output_dir=f"outputs/{run_name}",
        run_name=run_name,
        learning_rate=1e-6,
        lr_scheduler_type="constant_with_warmup",
        warmup_steps=10,
        num_train_epochs=1,
        max_steps=500,
        bf16=False,
        max_grad_norm=0.001,
        num_iterations=1,
        # max_concurrent=0,             # synchronise generation
        # num_batches_ahead=0,
        max_prompt_length=1024,
        max_completion_length=2048,
        per_device_train_batch_size=2,
        num_generations=8,
        gradient_accumulation_steps=4,
        gradient_checkpointing=False,
        save_strategy="steps",
        save_steps=500,
        save_only_model=True,
        logging_steps=1,
        log_on_each_node=False,
        log_completions=True,
        report_to=None,
        do_train=True
    )
training_args.num_iterations = 1
training_args.per_device_train_batch_size = 1
training_args.num_generations = 8

In [ ]:
trl vllm-serve \
    --model meta-llama/Llama-3.2-1B-Instruct \
    --dtype bfloat16 \
    --tensor-parallel-size 1 \
    --port 8000

In [ ]:
trainer = vf.GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    env=vf_env,
    args=training_args,
)
trainer.train()